Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).

The classes are ordered and not balanced (e.g. there are many more normal wines than excellent or poor ones). Outlier detection algorithms could be used to detect the few excellent or poor wines. Also, we are not sure if all input variables are relevant. So it could be interesting to test feature selection methods.

In [2]:
%pip install ucimlrepo

  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-1.26.4:
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import numpy as np
import pandas as pd
import math
import matplotlib.text as plttxt

In [5]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)
X = wine_quality.data.features
y = wine_quality.data.targets

# metadata
print(wine_quality.metadata)

# variable information
print(wine_quality.variables)

df = pd.DataFrame(X)

{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

## Step 2: Inspect both datasets: check their shapes, columns, data types, missing values, duplicates, and quality score ranges.

In [6]:
# Combine physicochemical features and quality for the audit
data = pd.concat([X, y], axis=1)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Combined shape:", data.shape)
print("\nData types:\n", data.dtypes)
print("\nMissing values by column:\n", data.isna().sum())
print("\nDuplicate rows, including quality:", data.duplicated().sum())

quality_col = y.columns[0]
print(f"\nQuality range: {y[quality_col].min()} to {y[quality_col].max()}")
print("\nQuality score counts:\n", y[quality_col].value_counts().sort_index())

# Check whether the repository exposes wine color in its original table
original = getattr(wine_quality.data, "original", None)

if isinstance(original, pd.DataFrame) and "color" in original.columns:
    print("\nWine type counts:\n", original["color"].value_counts(dropna=False))
else:
    print("\nWine color is not available in the original table.")

Feature shape: (6497, 11)
Target shape: (6497, 1)
Combined shape: (6497, 12)

Data types:
 fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

Missing values by column:
 fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

Duplicate rows, including quality: 1179

Quality range: 3 to 9

Quality score counts:
 quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9 

In [7]:
print("Missing values in combined data:", data.isna().sum().sum())
print("Duplicate rows, including quality:", data.duplicated().sum())
print("\nQuality score counts:")
print(y[quality_col].value_counts().sort_index().to_string())

Missing values in combined data: 0
Duplicate rows, including quality: 1179

Quality score counts:
quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5


## Step 2: Clarify the target and inputs

The target is `quality`. The 11 physicochemical measurements in `X` are the model features. Add the `color` field from the original table so red and white wines remain distinguishable in the combined dataset. Keep a full copy and a separate exact-deduplicated copy for later comparison.

In [8]:
original = wine_quality.data.original
quality_col = y.columns[0]
feature_cols = list(X.columns)

# Verify that the color labels come from the same source rows as X and y.
assert len(X) == len(y) == len(original), 'Feature, target, and source row counts differ.'
assert 'color' in original.columns, 'The original table has no color column.'
if quality_col in original.columns:
    assert original[quality_col].reset_index(drop=True).equals(y[quality_col].reset_index(drop=True)), 'Original quality values do not align with y.'

# Keep all rows and add the target and wine type.
wine_data_all = X.reset_index(drop=True).copy()
wine_data_all[quality_col] = y[quality_col].reset_index(drop=True)
wine_data_all['color'] = original['color'].reset_index(drop=True)

# Model inputs are the physicochemical measurements plus wine type; quality is the target.
X_model = wine_data_all[feature_cols + ['color']]
y_model = wine_data_all[quality_col]

# Preserve red and white subsets for separate-model comparisons later.
red_wines = wine_data_all.loc[wine_data_all['color'].eq('red')].copy()
white_wines = wine_data_all.loc[wine_data_all['color'].eq('white')].copy()

# Keep the original rows and make a separate exact-deduplicated comparison copy.
wine_data_unique = wine_data_all.drop_duplicates().reset_index(drop=True)

print('Target:', quality_col)
print('Physicochemical features:', feature_cols)
print('Model input columns:', list(X_model.columns))
print('Target shape:', y_model.shape)
print('Rows by wine type:', wine_data_all['color'].value_counts().to_dict())
print('Rows before duplicate removal:', len(wine_data_all))
print('Exact duplicates including color:', wine_data_all.duplicated().sum())
print('Rows in unique comparison copy:', len(wine_data_unique))
print('Red subset shape:', red_wines.shape)
print('White subset shape:', white_wines.shape)

Target: quality
Physicochemical features: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Model input columns: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'color']
Target shape: (6497,)
Rows by wine type: {'white': 4898, 'red': 1599}
Rows before duplicate removal: 6497
Exact duplicates including color: 1177
Rows in unique comparison copy: 5320
Red subset shape: (1599, 13)
White subset shape: (4898, 13)


## Step 3: Explore the data

Explore the target distribution, compare red and white wines, and inspect physicochemical feature distributions and their rank correlations with quality. These summaries can reveal patterns and questions for later analysis; correlations alone do not establish cause and effect.

In [ ]:
# Quality score distribution overall and for each wine type.
print('Quality counts overall:')
display(wine_data_all[quality_col].value_counts().sort_index().rename('count').to_frame())

print('Quality summary by wine type:')
display(wine_data_all.groupby('color')[quality_col].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2))

print('Quality counts by wine type:')
display(pd.crosstab(wine_data_all[quality_col], wine_data_all['color']).reindex(columns=['red', 'white'], fill_value=0))

# Inspect feature scale, spread, and skew to guide later preprocessing and modeling choices.
feature_summary = wine_data_all[feature_cols].describe().T
feature_summary['skew'] = wine_data_all[feature_cols].skew()
print('Physicochemical feature summary:')
display(feature_summary.round(3))

# Spearman correlation is used because quality is an ordered score.
spearman_by_color = {}
for color in ['red', 'white']:
    subset = wine_data_all.loc[wine_data_all['color'].eq(color), feature_cols + [quality_col]]
    spearman_by_color[color] = subset.corr(method='spearman')[quality_col].drop(quality_col)

spearman_results = pd.DataFrame({
    'all_wines': wine_data_all[feature_cols + [quality_col]].corr(method='spearman')[quality_col].drop(quality_col),
    'red': spearman_by_color['red'],
    'white': spearman_by_color['white'],
})
spearman_results = spearman_results.loc[spearman_results['all_wines'].abs().sort_values(ascending=False).index]
print('Spearman correlations between features and quality (descriptive, not causal):')
display(spearman_results.round(3))

In [ ]:
import matplotlib.pyplot as plt

# Compare score frequencies, using percentages in the red/white chart because group sizes differ.
quality_counts = wine_data_all[quality_col].value_counts().sort_index()
quality_pct_by_color = pd.crosstab(wine_data_all[quality_col], wine_data_all['color'], normalize='columns').reindex(columns=['red', 'white']) * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
quality_counts.plot(kind='bar', ax=axes[0], color='slateblue')
axes[0].set(title='Wine quality counts', xlabel='Quality score', ylabel='Number of wines')
quality_pct_by_color.plot(kind='bar', ax=axes[1], color=['firebrick', 'goldenrod'])
axes[1].set(title='Quality distribution within each wine type', xlabel='Quality score', ylabel='Percent of wine type')
axes[1].legend(title='Wine type')
plt.tight_layout()
plt.show()

# Compare feature shapes across wine types with density histograms, so group size does not dominate.
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
for ax, feature in zip(axes.flat, feature_cols):
    for color, color_name in [('red', 'firebrick'), ('white', 'goldenrod')]:
        values = wine_data_all.loc[wine_data_all['color'].eq(color), feature]
        ax.hist(values, bins=30, density=True, alpha=0.45, color=color_name, label=color)
    ax.set_title(feature)
    ax.set_ylabel('Density')
    ax.legend()
for ax in axes.flat[len(feature_cols):]:
    ax.set_visible(False)
fig.suptitle('Physicochemical feature distributions by wine type', y=1.01)
plt.tight_layout()
plt.show()

## Step 4: Set up the evaluation split

Set aside a test set now and keep it untouched during model fitting, preprocessing choices, and feature selection. Because identical input profiles occur more than once, use a grouped split so copies of the same profile stay on the same side. Stratify approximately by quality score while forming five folds, then reserve one fold (about 20%) as the test set.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Group identical predictor profiles (including wine color) to prevent duplicate leakage.
group_cols = feature_cols + ['color']
groups = pd.MultiIndex.from_frame(wine_data_all[group_cols]).factorize()[0]

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(X_model, y_model, groups=groups))

train_data = wine_data_all.iloc[train_idx].copy()
test_data = wine_data_all.iloc[test_idx].copy()

# Keep predictors and target separate for modeling.
X_train = train_data[group_cols].copy()
y_train = train_data[quality_col].copy()
X_test = test_data[group_cols].copy()
y_test = test_data[quality_col].copy()

# Verify that each predictor profile belongs to only one partition.
train_groups = set(groups[train_idx])
test_groups = set(groups[test_idx])
assert train_groups.isdisjoint(test_groups), 'A predictor profile appears in both train and test.'
assert len(train_data) + len(test_data) == len(wine_data_all)

print(f'Train rows: {len(train_data):,} ({len(train_data) / len(wine_data_all):.1%})')
print(f'Test rows:  {len(test_data):,} ({len(test_data) / len(wine_data_all):.1%})')
print(f'Unique predictor profiles: {len(set(groups)):,}')
print('Train rows by wine type:', train_data['color'].value_counts().to_dict())
print('Test rows by wine type:', test_data['color'].value_counts().to_dict())

# Inspect class coverage only. Do not use test performance for feature selection or tuning.
split_quality_counts = pd.concat([
    y_train.value_counts().rename('train_count'),
    y_test.value_counts().rename('test_count'),
], axis=1).fillna(0).astype(int).sort_index()
print('Quality score counts in each partition:')
display(split_quality_counts)